In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
load_dotenv()

/Users/rahultiwari/Documents/aziz_ai_engineering/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_10152/607174786.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
# pip install langchain_community

In [3]:
# pip install pypdf

In [4]:
# you need data --> pdf ?? pypdf 
loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")

pages = loader.load()

In [5]:
len(pages)

14

In [6]:
print(  pages[0].page_content)

BAJAJ FINANCE LIMITED
 Helpdesk Agent Knowledge Base
 Prose Reference Edition — FY 2024–25
 Document Type: Internal Training & Reference Manual
 Coverage: Personal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy
 Intended Users: Helpdesk Agents · Branch Executives · Collections Team
 Classification: CONFIDENTIAL — For Internal Use Only
 Version: v1.0 Prose Edition — May 2025
This document is the prose-format reference edition of the Bajaj Finance Helpdesk Knowledge Base. All
policy information is presented in descriptive paragraph form to support agent training, onboarding, and
the BajajBot AI assistant knowledge base. For structured lookup tables, refer to the companion Policy
Reference Document v4.0.
Section 1 — Personal Loan: Eligibility, Rates & Charges
1.1 Who Can Apply for a Bajaj Finance Personal Loan
Bajaj Finance personal loans are designed for both salaried employees and self-employed
professionals. However, all eligibility conditions must be satisfied simultaneou

In [7]:
print(pages[2].page_content)

Every personal loan at Bajaj Finance carries a processing fee, which ranges from 1.5 to 3.5 percent of
the loan amount plus GST, subject to a minimum of Rs 1,999. This fee is deducted upfront from the
disbursed amount and is non-refundable even if the loan is cancelled within the cooling-off period.
If an EMI payment is missed or delayed, a late payment charge of 2 percent per month is applied on the
overdue EMI amount, starting from day one of the default. This charge compounds monthly and can
significantly increase the outstanding amount if not addressed promptly.
When a NACH or ECS auto-debit mandate fails due to insufficient balance in the registered bank
account, a bounce charge of Rs 1,000 plus GST is levied per bounce event. This is in addition to any
late payment charges that may apply.
Customers who wish to close their loan early have the option of full prepayment. If the loan is closed
within the first 12 months, a prepayment penalty of 4 percent of the outstanding principal 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=350,
                                          separators=["\n\n", "\n", " ", ""],
                                          chunk_overlap=64 # character overlap
                                          )


chunks = splitter.split_documents(pages)

In [9]:
# print(  chunks[1].page_content)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9317.66it/s]


In [10]:
chunk_text = [chunk.page_content for chunk in chunks]
chunk_meta = [chunk.metadata for chunk in chunks]

In [11]:
chunk_vectors = model.encode(
    chunk_text,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True # makes cosin --> faster
)

Batches: 100%|██████████| 5/5 [00:00<00:00, 14.31it/s]


In [12]:
# print( chunks[2].page_content)

In [14]:
store = {
    'vectors':chunk_vectors,
    'text':chunk_text,
    'meta':chunk_meta,
    'model':'all-MiniLM-L6-v2'
}

import pickle

with open("bfl_store.pkl", "wb") as f:
    pickle.dump(store, f)